In [14]:
import math

from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math

from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

In [19]:
class ParametricMethod:
    def __init__(self,filename,p,logTrue=False,distribution=stats.gamma):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.distribution = distribution
        self.dataframe = pd.DataFrame()

    def generateOutput(self):
        self._readArff()
        for v in range(2,70):
            self._distanceMetric(v)
            self._generateArray()
            if self.logTrue:
                self.arr = np.log(self.arr)
                self.arr[~np.isfinite(self.arr)] = 0
            params = self.distribution.fit(self.arr) # fit params for gamma distribution
            posNeg4 = []
            spaceStep4 = np.linspace(0,.99,30) # threshold from 0 to .99, 30 samples
            for e in spaceStep4:
                if len(params) == 3:
                    newArr = self.arr > self.distribution.ppf(e,params[0],loc=params[1], scale=params[2]) # if arr value is outside threshold add to new array
                else:
                    newArr = self.arr > self.distribution.ppf(e,loc=params[0], scale=params[1]) # if arr value is outside threshold add to new array
                posNeg4.append([((self.y[newArr] == 1).sum() / (self.y == 1).sum()), (self.y[newArr] != 1).sum()/ ((self.y != 1).sum())]) # True positive rate, false positive rate

            posNeg4 = np.array(posNeg4)
            arrtest1, arrtest2 = np.split(posNeg4, 2,axis=1) # split the array
            self.tots += [auc(arrtest2, arrtest1)] # return the area under the curve

        maxValue, kIndex = self._printResults(self.tots)
        return maxValue, kIndex

    def probPlots(self,v):
        self._readArff()
        self._distanceMetric(v)
        self._generateArray()
        if self.logTrue:
            self.arr = np.log(self.arr)
            self.arr[~np.isfinite(self.arr)] = 0
        params = self.distribution.fit(self.arr) # fit params for gamma distribution
        if len(params) == 3:
             results = probplot(self.arr,dist=self.distribution,sparams=(params[0],params[1],params[2]),rvalue=True)
        else:
             results = probplot(self.arr,dist=self.distribution,sparams=(params[0],params[1]),rvalue=True)

        self.tots += [results[1][2]]

        return self.tots

    def _distanceMetric(self,n):
        #find the nearestNeighbors
        nn = NearestNeighbors(n_neighbors=n,p=self.p)
        nn.fit(self.X, self.y)
        #return the dist of each and the nearest neighbors
        self.distance, knn = nn.kneighbors(self.X)  # returns N index neighbors including self

    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _generateArray(self):
        self.arr = []
        #returns an array based on the median and max values
        for x in self.distance:  # finds the distance away from that point (index 0)
            self.arr += [np.max(x)]

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        print(max(newarr),newarr.index(max(newarr))+2) #print the max values, the k value, and the array
        return max(newarr),newarr.index(max(newarr))+2

In [3]:
positively_skewed_distributions = [
    stats.expon,
    stats.chi2,
    stats.gamma,
    stats.weibull_min,
    stats.lognorm,
    stats.invgauss,
    stats.rayleigh,
    stats.wald,
    stats.pareto,
    stats.levy,
    stats.nakagami,
    stats.logistic,
    stats.powerlaw,
    stats.skewnorm
]

folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]


In [4]:
dict = {}
for z in folder_structure_1d:
    print(z)
    dict[z] = {}
    for i in positively_skewed_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i)
        maxVal, indexMax = holder.generateOutput()
        dict[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict)

# Print DataFrame


semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff
0.6761145800501458 2
0.6763770930764142 2
0.6764974884502787 2
0.6760776663741966 2
0.6758756349862142 2
0.6769205759669252 2
0.6759453450434872 2
0.6763670128033665 2
0.6760085242196305 2
0.6768747178233426 2
0.6762105556076133 2
0.6735908204206454 2
0.6746039588497698 2
0.6751869109784112 2
semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff
0.7565653350310361 35
0.7611013846888429 46
0.7610814897342033 41
0.7596590004774788 44
0.7599375298424319 45
0.760335428935222 45
0.7598778449785134 45
0.7604846410950183 44
0.7606636956867737 35
0.7610715422568837 37
0.7602459016393441 38
0.761499283781633 29
0.7608725927104887 45
0.7620265000795797 45
semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff
0.5564958435768157 69
0.5568532803450144 69
0.5571775126046918 69
0.5578806669027877 68
0.5563024761448393 69
0.5564209706654443 67
0.5575811752573024 69
0.5575609921246719 69
0.5564964946456102 69
0.5580538512021334 69


In [5]:
df.head()

,semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff,semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff,semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff,semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff,semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff,semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff,semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff,semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff,semantic/Pima/Pima_withoutdupl_norm_35.arff,semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff,semantic/Stamps/Stamps_withoutdupl_norm_09.arff,semantic/Wilt/Wilt_withoutdupl_norm_05.arff
<scipy.stats._continuous_distns.expon_gen object at 0x0000015138120350>,"[0.6761145800501458, 2]","[0.7565653350310361, 35]","[0.5564958435768157, 69]","[0.6950833333333333, 69]","[0.7703788748564867, 25]","[0.7212182687598628, 14]","[0.8703679833596352, 69]","[0.716907596371882, 6]","[0.7337910447761193, 64]","[0.6494480401987319, 51]","[0.9166927654243658, 63]","[0.5602741817449851, 3]"
<scipy.stats._continuous_distns.chi2_gen object at 0x00000151381129D0>,"[0.6763770930764142, 2]","[0.7611013846888429, 46]","[0.5568532803450144, 69]","[0.7001666666666667, 66]","[0.7887485648679678, 40]","[0.722259822060184, 14]","[0.8705970694646895, 69]","[0.7381660997732425, 6]","[0.7365335820895523, 66]","[0.650278170965237, 49]","[0.9193026411942791, 67]","[0.5513837026220666, 2]"
<scipy.stats._continuous_distns.gamma_gen object at 0x0000015137E1AF90>,"[0.6764974884502787, 2]","[0.7610814897342033, 41]","[0.5571775126046918, 69]","[0.7001666666666667, 66]","[0.7847301951779564, 26]","[0.722259822060184, 14]","[0.8697028104709015, 68]","[0.7381660997732425, 6]","[0.7359141791044777, 69]","[0.6500094710534449, 49]","[0.9193026411942791, 67]","[0.5606840129167187, 3]"
<scipy.stats._continuous_distns.weibull_min_gen object at 0x00000151381236D0>,"[0.6760776663741966, 2]","[0.7596590004774788, 44]","[0.5578806669027877, 68]","[0.7016388888888889, 68]","[0.7898966704936855, 26]","[0.7036376992980354, 6]","[0.870827360229367, 68]","[0.7414965986394557, 4]","[0.7357313432835821, 68]","[0.6504375532452259, 41]","[0.9186240734941016, 68]","[0.5558884338052292, 3]"
<scipy.stats._continuous_distns.lognorm_gen object at 0x0000015138172050>,"[0.6758756349862142, 2]","[0.7599375298424319, 45]","[0.5563024761448393, 69]","[0.6989444444444444, 69]","[0.791044776119403, 25]","[0.7219392787179627, 14]","[0.8712933627270281, 69]","[0.7385912698412699, 6]","[0.7358059701492536, 63]","[0.6508622192610126, 40]","[0.9206075790792358, 64]","[0.5624772908325756, 2]"


In [6]:
df.to_csv('out.csv')

In [4]:
literature_dataset_paths = [
    "literature/ALOI/ALOI_withoutdupl_norm.arff",
    "literature/Glass/Glass_withoutdupl_norm.arff",
    "literature/Ionosphere/Ionosphere_withoutdupl_norm.arff",
    "literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff",
    "literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff",
    "literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff",
    "literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff",
    "literature/Waveform/Waveform_withoutdupl_norm_v10.arff",
    "literature/WBC/WBC_withoutdupl_norm_v10.arff",
    "literature/WDBC/WDBC_withoutdupl_norm_v10.arff",
    "literature/WPBC/WPBC_withoutdupl_norm.arff"
]

positively_skewed_distributions = [
    stats.expon,
    stats.chi2,
    stats.gamma,
    stats.weibull_min,
    stats.lognorm,
    stats.invgauss,
    stats.rayleigh,
    stats.wald,
    stats.pareto,
    stats.levy,
    stats.nakagami,
    stats.logistic,
    stats.powerlaw,
    stats.skewnorm
]



In [9]:
dict2 = {}
for z in literature_dataset_paths:
    print(z)
    dict2[z] = {}
    for i in positively_skewed_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i)
        maxVal, indexMax = holder.generateOutput()
        dict2[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict2)

literature/ALOI/ALOI_withoutdupl_norm.arff
0.7427896248395957 3
0.7439105914778037 3
0.7447408432943208 2
0.7439272574062171 2
0.7453912632536245 3
0.7455685613926408 3
0.742619665508327 3
0.7435351248180003 3
0.7451714856375874 3
0.7450699712169612 2
0.7447039628512452 3
0.7380774820137765 3
0.7451638637714032 3
0.7433926152511774 3
literature/Glass/Glass_withoutdupl_norm.arff
0.8712737127371273 2
0.8780487804878049 2
0.8804878048780487 2
0.8745257452574526 10
0.8753387533875339 9
0.8845528455284554 2
0.8753387533875339 2
0.8783197831978319 2
0.8794037940379403 2
0.8845528455284553 2
0.8799457994579946 2
0.8718157181571815 10
0.8739837398373984 10
0.8772357723577238 2
literature/Ionosphere/Ionosphere_withoutdupl_norm.arff
0.9013932980599647 2
0.9006349206349207 2
0.9008641975308642 2
0.900952380952381 2
0.8962962962962963 5
0.900758377425044 2
0.900546737213404 2
0.9005996472663139 2
0.9013932980599647 2
0.9014814814814816 2
0.9005291005291005 2
0.9022222222222221 2
0.8989241622574956

In [10]:
df.head()

,literature/ALOI/ALOI_withoutdupl_norm.arff,literature/Glass/Glass_withoutdupl_norm.arff,literature/Ionosphere/Ionosphere_withoutdupl_norm.arff,literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff,literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff,literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff,literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff,literature/Waveform/Waveform_withoutdupl_norm_v10.arff,literature/WBC/WBC_withoutdupl_norm_v10.arff,literature/WDBC/WDBC_withoutdupl_norm_v10.arff,literature/WPBC/WPBC_withoutdupl_norm.arff
<scipy.stats._continuous_distns.expon_gen object at 0x0000015138120350>,"[0.7427896248395957, 3]","[0.8712737127371273, 2]","[0.9013932980599647, 2]","[0.949934412372425, 69]","[0.9929577464788732, 13]","[0.9908560113728677, 12]","[0.8417307692307692, 5]","[0.7854636553993419, 62]","[0.9882629107981221, 10]","[0.984873949579832, 42]","[0.5323376074397632, 14]"
<scipy.stats._continuous_distns.chi2_gen object at 0x00000151381129D0>,"[0.7439105914778037, 3]","[0.8780487804878049, 2]","[0.9006349206349207, 2]","[0.9688893932753115, 69]","[1.0, 38]","[0.9841287571080423, 6]","[0.8447692307692307, 5]","[0.7855160035895901, 66]","[0.9983568075117372, 30]","[0.9896358543417368, 56]","[0.5315626320980695, 26]"
<scipy.stats._continuous_distns.gamma_gen object at 0x0000015137E1AF90>,"[0.7447408432943208, 2]","[0.8804878048780487, 2]","[0.9008641975308642, 2]","[0.9674649886252166, 69]","[0.9999999999999999, 8]","[0.9826056051990251, 12]","[0.8201538461538461, 5]","[0.785513012264433, 66]","[0.9985915492957746, 40]","[0.9887955182072827, 68]","[0.5315626320980695, 26]"
<scipy.stats._continuous_distns.weibull_min_gen object at 0x00000151381236D0>,"[0.7439272574062171, 2]","[0.8745257452574526, 10]","[0.900952380952381, 2]","[0.9676234529250934, 69]","[1.0, 8]","[0.982245125913891, 9]","[0.8392307692307692, 5]","[0.7847442416990726, 67]","[0.9981220657276997, 26]","[0.9893557422969189, 64]","[0.5329716781738761, 12]"
<scipy.stats._continuous_distns.lognorm_gen object at 0x0000015138172050>,"[0.7453912632536245, 3]","[0.8753387533875339, 9]","[0.8962962962962963, 5]","[0.9668039989146996, 65]","[0.9999999999999998, 8]","[0.9867536555645817, 12]","[0.8483846153846154, 5]","[0.7854875860005983, 66]","[0.9978873239436619, 69]","[0.9868347338935577, 48]","[0.5306467521487952, 20]"


In [11]:
df.to_csv('literature.csv')

In [5]:
normal_like_distributions = [
    stats.norm,
    stats.t,
    stats.laplace,
    stats.cauchy,
    stats.logistic,
    stats.skewnorm,
]

folder_structure_1d = [
    "semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff",
    "semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",
    "semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff",
    "semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff",
    "semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff",
    "semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff",
    "semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff",
    "semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff",
    "semantic/Pima/Pima_withoutdupl_norm_35.arff",
    "semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff",
    "semantic/Stamps/Stamps_withoutdupl_norm_09.arff",
    "semantic/Wilt/Wilt_withoutdupl_norm_05.arff"
]


In [24]:
dict3 = {}
for z in folder_structure_1d:
    print(z)
    dict3[z] = {}
    for i in normal_like_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i,logTrue=True)
        maxVal, indexMax = holder.generateOutput()
        dict3[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict3)

semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff
0.6757414679717071 2
0.677327478256425 2
0.6762253210779929 2
0.6755161525727413 2
0.6770895270222304 2
0.6766126307241044 2
semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff
0.7624045042177304 44
0.7613202291898775 47
0.7621558172847367 34
0.7600469520929493 45
0.7607034855960528 47
0.7598778449785135 35
semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff
0.5569646131088796 69
0.556473056169007 69
0.5567738499520812 68
0.5580759875411475 69
0.5564307366973624 69
0.5568604421017543 69
semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff
0.7006666666666668 69
0.7006666666666668 69
0.6973611111111111 69
0.6992777777777778 68
0.6984166666666667 68
0.7005277777777777 68
semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff
0.7881745120551091 26
0.7881745120551091 26
0.7904707233065443 25
0.7973593570608497 26
0.7898966704936854 26
0.7881745120551091 26
semantic/InternetAds/InternetAds_withoutdupl_norm_19.a

In [25]:
df.to_csv('logtransforSem.csv')

In [29]:
normal_like_distributions = [
    stats.norm,
    stats.t,
    stats.laplace,
    stats.cauchy,
    stats.logistic,
    stats.skewnorm,
]

literature_dataset_paths = [
    "literature/ALOI/ALOI_withoutdupl_norm.arff",
    "literature/Glass/Glass_withoutdupl_norm.arff",
    "literature/Ionosphere/Ionosphere_withoutdupl_norm.arff",
    "literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff",
    "literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff",
    "literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff",
    "literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff",
    "literature/Waveform/Waveform_withoutdupl_norm_v10.arff",
    "literature/WBC/WBC_withoutdupl_norm_v10.arff",
    "literature/WDBC/WDBC_withoutdupl_norm_v10.arff",
    "literature/WPBC/WPBC_withoutdupl_norm.arff"
]

In [30]:
dict3 = {}
for z in literature_dataset_paths:
    print(z)
    dict3[z] = {}
    for i in normal_like_distributions:
        print(i)
        holder = ParametricMethod(z,1,distribution=i,logTrue=True)
        maxVal, indexMax = holder.generateOutput()
        dict3[z][i] = [maxVal,indexMax]

df = pd.DataFrame.from_dict(dict3)

literature/ALOI/ALOI_withoutdupl_norm.arff
0.7454481510954334 3
0.7456501305493123 3
0.7454551032867808 2
0.7455418020146249 2
0.7452323570090958 3
0.7450285204157208 3
literature/Glass/Glass_withoutdupl_norm.arff
0.8723577235772358 10
0.8758807588075881 2
0.8804878048780489 2
0.8758807588075881 11
0.8764227642276423 2
0.8761517615176151 10
literature/Ionosphere/Ionosphere_withoutdupl_norm.arff
0.9089770723104056 2
0.9089770723104056 2
0.9068606701940036 2
0.9055908289241622 2
0.9067548500881835 2
0.908042328042328 2
literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff
0.9680974891991735 69
0.9684314799741197 68
0.967350405944107 69
0.9626280967587084 69
0.9674127063636173 69
0.9665843821092396 69
literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff
1.0 19
0.9988262910798122 6
0.9982394366197185 31
1.0 8
0.9999999999999999 15
0.9999999999999998 8
literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff
0.9824380584890333 9
0.9825319861900893 10
0.9838799756295694 14
0.98727

In [31]:
df.to_csv('logtransforLit.csv')

In [26]:
invgaussK = [2, 45, 67, 69, 26, 14, 68, 6, 63, 40, 64, 2]
exponK = [2, 35, 69, 69, 25, 14, 69, 6, 64, 51, 63, 3]
cauchy = [2, 35, 69, 68, 26, 14, 69, 6, 67, 46, 65, 3]

In [27]:
dict = {}
for z in range(len(folder_structure_1d)):
    print(folder_structure_1d[z])
    holder = ParametricMethod(folder_structure_1d[z],1,distribution=stats.skewnorm,logTrue=True)
    rvalue = holder.probPlots(cauchy[z])
    print(pow(rvalue[0],2))
    dict[z] = pow(rvalue[0],2)
df = pd.DataFrame.from_dict([dict])

semantic/Annthyroid/Annthyroid_withoutdupl_norm_07.arff
0.974761909815642
semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff
0.9948632153883507
semantic/Cardiotocography/Cardiotocography_withoutdupl_norm_22.arff
0.994444617791968
semantic/HeartDisease/HeartDisease_withoutdupl_norm_44.arff
0.993107115572747
semantic/Hepatitis/Hepatitis_withoutdupl_norm_16.arff
0.9644613023436949
semantic/InternetAds/InternetAds_withoutdupl_norm_19.arff
0.9811160884426221
semantic/PageBlocks/PageBlocks_withoutdupl_norm_09.arff
0.9962380645955136
semantic/Parkinson/Parkinson_withoutdupl_norm_75.arff
0.9692348961078574
semantic/Pima/Pima_withoutdupl_norm_35.arff
0.9906327312753171
semantic/SpamBase/SpamBase_withoutdupl_norm_40.arff
0.8393856920612278
semantic/Stamps/Stamps_withoutdupl_norm_09.arff
0.9926552788292422
semantic/Wilt/Wilt_withoutdupl_norm_05.arff
0.9795205273662072


In [29]:
df.head()
df.to_csv("skewnormR^2.csv")

In [33]:
exponK2 = [3, 2, 2, 69, 13, 12, 5, 62, 10, 42, 14]
invgaussK2 = [3, 2, 2, 69, 8, 12, 5, 66, 4, 25, 19]
cauchy2 = [3, 2, 2, 68, 6, 10, 5, 64, 23, 46, 20]

In [34]:
dict = {}
for z in range(len(literature_dataset_paths)):
    print(literature_dataset_paths[z])
    holder = ParametricMethod(literature_dataset_paths[z],1,distribution=stats.t,logTrue=True)
    rvalue = holder.probPlots(cauchy2[z])
    print(pow(rvalue[0],2))
    dict[z] = pow(rvalue[0],2)
df = pd.DataFrame.from_dict([dict])

literature/ALOI/ALOI_withoutdupl_norm.arff
0.988644548046837
literature/Glass/Glass_withoutdupl_norm.arff
0.9137893331102086
literature/Ionosphere/Ionosphere_withoutdupl_norm.arff
0.9835891253379914
literature/KDDCup99/KDDCup99_withoutdupl_norm_idf.arff
0.8837955889056123
literature/Lymphography/Lymphography_withoutdupl_norm_idf.arff
0.9671292478777089
literature/PenDigits/PenDigits_withoutdupl_norm_v10.arff
0.9766378133318113
literature/Shuttle/Shuttle_withoutdupl_norm_v10.arff
0.992923042478918
literature/Waveform/Waveform_withoutdupl_norm_v10.arff
0.9911635128071926
literature/WBC/WBC_withoutdupl_norm_v10.arff
0.8370042761807102
literature/WDBC/WDBC_withoutdupl_norm_v10.arff
0.8559968190715588
literature/WPBC/WPBC_withoutdupl_norm.arff
0.8997236926874964


In [35]:
df.to_csv("tLitR^2.csv")